# 03 — Training

Fine-tune encoder models and run LLM baselines.  
All runs use the same hyperparameters — the only variable is which density column is used for weighting.

**Sections**
1. Configuration
2. Define experiments (dataset × density column)
3. Fine-tuning runs
4. LLM baselines
5. Results summary

In [1]:
import sys, os
if os.getcwd().endswith('notebooks'):
    os.chdir('..')

import json
import yaml
import pandas as pd

from src.training import TrainingConfig, train, run_baselines

c:\Users\Alexandre\miniconda3\envs\faiss2\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 1. Configuration

Edit `TrainingConfig` fields below to change model, hyperparameters, or output paths.  
All experiments in this notebook share the same config — only `density_column` varies per run.

In [2]:
CONFIG_PATH = "configs/datasets.yaml"
with open(CONFIG_PATH) as f:
    cfg = yaml.safe_load(f)

EMBEDDINGS_ROOT   = cfg["embedding"]["output_root"]
PREPROCESSED_ROOT = cfg["preprocessing"]["output_root"]
MODEL_NAME        = cfg["embedding"]["models"][0]
MODEL_SLUG        = MODEL_NAME.replace("/", "_")
K_VALUES          = cfg["embedding"]["k_values"]

# Russian is reference-only: not used for fine-tuning, only for density + eval
TRAIN_DATASETS    = cfg.get("train_datasets", ["toxigen"])

train_cfg = cfg["training"]

base_config = TrainingConfig(
    model_id     = train_cfg["models"][0],
    batch_size   = train_cfg["batch_size"],
    learning_rate= train_cfg["learning_rate"],
    num_epochs   = train_cfg["epochs"],
    max_length   = train_cfg["max_length"],
    random_state = train_cfg["random_state"],
    output_root  = train_cfg["output_root"],
)

print("Model          :", base_config.model_id)
print("Epochs         :", base_config.num_epochs)
print("LR             :", base_config.learning_rate)
print("Train datasets :", TRAIN_DATASETS)
print("Output         :", base_config.output_root)

Model          : answerdotai/ModernBERT-base
Epochs         : 1
LR             : 2e-5
Train datasets : ['toxigen']
Output         : outputs/3_training


## 2. Define Experiments

Each experiment is `(dataset, density_column)`.  
- `density_column = None` → train without weighting (baseline encoder)
- `density_column = 'density_k5_ratio'` → weight by Russian/All ratio at K=5 on raw embeddings
- `density_column = 'density_pca_k5_ratio'` → same but PCA space

Edit the list below to add/remove experiments.

In [3]:
# Build experiment list: one fine-tune per (train_dataset × density_column)
# Russian is excluded — it is the reference group used for density, not for training.
density_columns = [None]  # baseline: no weighting
for k in K_VALUES:
    density_columns.append(f"density_k{k}_ratio")       # raw space
    density_columns.append(f"density_pca_k{k}_ratio")   # PCA space

experiments = [
    {"dataset": ds, "density_column": dc}
    for ds in TRAIN_DATASETS
    for dc in density_columns
]

print(f"{len(experiments)} fine-tunes planned:")
for ex in experiments:
    tag = f"{ex['dataset']}__{ex['density_column'] or 'no_density'}"
    print(f"  {tag}")

7 fine-tunes planned:
  toxigen__no_density
  toxigen__density_k5_ratio
  toxigen__density_pca_k5_ratio
  toxigen__density_k100_ratio
  toxigen__density_pca_k100_ratio
  toxigen__density_k1000_ratio
  toxigen__density_pca_k1000_ratio


## 3. Fine-Tuning Runs

Each run loads `outputs/2_embeddings/{dataset}/{model_slug}/densities.csv`,  
fine-tunes the model, and saves the checkpoint + `metrics.json` to `outputs/3_training/`.

In [4]:
all_metrics = {}

for ex in experiments:
    ds = ex["dataset"]
    dc = ex["density_column"]
    dataset_tag = ds

    density_csv = os.path.join(EMBEDDINGS_ROOT, ds, MODEL_SLUG, "densities.csv")
    if not os.path.exists(density_csv):
        print(f"[SKIP] {density_csv} not found")
        continue

    # Clone config and set density column for this run
    from dataclasses import replace
    run_config = TrainingConfig(
        model_id      = base_config.model_id,
        batch_size    = base_config.batch_size,
        learning_rate = base_config.learning_rate,
        num_epochs    = base_config.num_epochs,
        max_length    = base_config.max_length,
        random_state  = base_config.random_state,
        output_root   = base_config.output_root,
        density_column= dc,
    )

    run_name = run_config.run_name(dataset_tag)
    metrics_path = os.path.join(run_config.output_dir(dataset_tag), "metrics.json")

    if os.path.exists(metrics_path):
        print(f"[CACHE] {run_name} — loading existing metrics")
        with open(metrics_path) as f:
            all_metrics[run_name] = json.load(f)
        continue

    print(f"\n{'='*60}")
    print(f"Training: {run_name}")
    print(f"{'='*60}")
    metrics = train(density_csv=density_csv, dataset_tag=dataset_tag, config=run_config)
    all_metrics[run_name] = metrics

print("\nAll fine-tuning runs complete.")


Training: toxigen__no_density


Loading weights: 100%|██████████| 136/136 [00:00<00:00, 12816.82it/s]
[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.
[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': None, 'bos_token_id': None}.


Step,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1,Auc Roc
1000,0.263698,0.262840,0.887328,0.835532,0.821087,0.727079,0.771229,0.948657
2000,0.250951,0.217318,0.906934,0.844511,0.910577,0.713806,0.800274,0.967272
3000,0.225756,0.217438,0.911317,0.904947,0.794144,0.891609,0.840059,0.970630
4000,0.217854,0.194014,0.918629,0.910759,0.812925,0.894279,0.851664,0.974666
5000,0.216273,0.189035,0.924527,0.896974,0.867471,0.839283,0.853144,0.974900
6000,0.208163,0.181435,0.923590,0.914462,0.826562,0.895347,0.859580,0.976399
7000,0.154960,0.194244,0.925264,0.917024,0.828778,0.899771,0.862817,0.977097
8000,0.199428,0.190565,0.926061,0.882702,0.913506,0.791915,0.848376,0.978170
9000,0.201271,0.180430,0.928532,0.886273,0.917859,0.797788,0.853622,0.979906
10000,0.158110,0.162856,0.934130,0.913114,0.877542,0.869108,0.873304,0.980427


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.94s/it]


Training Loss,Validation Loss,Step,Accuracy,Balanced Accuracy,Precision,Recall,F1,Auc Roc
0.153107,0.158605,12548,0.935166,0.915417,0.877278,0.874066,0.875669,0.981582


...

Step,Training Loss,Validation Loss,Accuracy,Balanced Accuracy,Precision,Recall,F1,Auc Roc
1000,0.258641,0.261030,0.888822,0.849389,0.799380,0.766819,0.782761,0.947534
2000,0.260635,0.209312,0.911297,0.856438,0.901354,0.741571,0.813693,0.967444
3000,0.238374,0.232909,0.903367,0.909231,0.759716,0.921510,0.832828,0.970809
4000,0.209389,0.203201,0.917055,0.911148,0.806006,0.898780,0.849868,0.973767
5000,0.191539,0.199145,0.919984,0.910739,0.818462,0.891381,0.853366,0.974691
6000,0.201817,0.177040,0.926440,0.910325,0.847118,0.876583,0.861598,0.976640
7000,0.163551,0.192150,0.924626,0.917086,0.826005,0.901297,0.862010,0.977559
8000,0.195586,0.191333,0.926121,0.883606,0.911214,0.794584,0.848912,0.977763
9000,0.204555,0.180464,0.927057,0.882366,0.920591,0.788787,0.849608,0.979779
10000,0.160438,0.161532,0.933513,0.913287,0.874072,0.870938,0.872502,0.980559


Writing model shards: 100%|██████████| 1/1 [00:05<00:00,  5.85s/it]


Training Loss,Validation Loss,Step,Accuracy,Balanced Accuracy,Precision,Recall,F1,Auc Roc
0.148423,0.157932,12548,0.935704,0.916250,0.877993,0.875515,0.876752,0.981743


Writing model shards: 100%|██████████| 1/1 [00:06<00:00,  6.02s/it]



All fine-tuning runs complete.


## 4. LLM Baselines

Requires Ollama running locally (`ollama serve`).  
Runs zero-shot, context, and few-shot classification on the Russian test split.

In [5]:
RUN_BASELINES = False  # set True to run (requires Ollama)

if RUN_BASELINES:
    # Use the Russian annotated test set
    russian_test_csv = os.path.join(PREPROCESSED_ROOT, "russian", "test.csv")
    if os.path.exists(russian_test_csv):
        baseline_results = run_baselines(
            test_csv=russian_test_csv,
            dataset_tag="russian_annotated",
            config=base_config,
        )
        print("Baseline results:")
        for mode, metrics in baseline_results.items():
            print(f"  {mode}: F1={metrics['f1']:.4f}  Acc={metrics['accuracy']:.4f}")
    else:
        print(f"Russian test CSV not found at {russian_test_csv}")
else:
    print("Baselines skipped (RUN_BASELINES=False).")

Baselines skipped (RUN_BASELINES=False).


## 5. Results Summary

In [6]:
if all_metrics:
    rows = []
    for run_name, m in all_metrics.items():
        parts = run_name.split("__", 1)
        rows.append({
            "run": run_name,
            "dataset": parts[0],
            "density": parts[1] if len(parts) > 1 else "n/a",
            "f1":               m.get("f1", float("nan")),
            "accuracy":         m.get("accuracy", float("nan")),
            "balanced_accuracy": m.get("balanced_accuracy", float("nan")),
            "auc_roc":          m.get("auc_roc", float("nan")),
        })

    summary = pd.DataFrame(rows).sort_values("f1", ascending=False)
    display(summary.style.format({
        "f1": "{:.4f}",
        "accuracy": "{:.4f}",
        "balanced_accuracy": "{:.4f}",
        "auc_roc": "{:.4f}",
    }))
else:
    print("No metrics collected yet — run the training cells first.")

,run,dataset,density,f1,accuracy,balanced_accuracy,auc_roc
1,toxigen__density_k5_ratio,toxigen,density_k5_ratio,0.8768,0.9357,0.9162,0.9817
2,toxigen__density_pca_k5_ratio,toxigen,density_pca_k5_ratio,0.8768,0.9357,0.9162,0.9817
3,toxigen__density_k100_ratio,toxigen,density_k100_ratio,0.8768,0.9357,0.9162,0.9817
4,toxigen__density_pca_k100_ratio,toxigen,density_pca_k100_ratio,0.8768,0.9357,0.9162,0.9817
5,toxigen__density_k1000_ratio,toxigen,density_k1000_ratio,0.8768,0.9357,0.9162,0.9817
6,toxigen__density_pca_k1000_ratio,toxigen,density_pca_k1000_ratio,0.8768,0.9357,0.9162,0.9817
0,toxigen__no_density,toxigen,no_density,0.8757,0.9352,0.9154,0.9816
